# CLIP text-embedding spread diagnostic for IFCB taxonomy — model comparison

**Question:** if we replace FineDiffusion's `LabelEmbedder` lookup table with a projection of **CLIP text embeddings of taxonomic strings**, does the text space *separate our 145 plankton species finely enough* to condition on them? And which adaptation of CLIP gives the best-separated text embeddings?

All encoders share the **same backbone — OpenAI CLIP ViT-B/16** — so the comparison isolates the *adaptation method* and the *training loss*:

| key | adaptation | trained on | loss | rank |
|---|---|---|---|---|
| `plain` | none (frozen) | web image–caption pairs | — | — |
| `pz` | **full fine-tune** | Planktonzilla | (its own recipe) | — |
| `e0c` | **LoRA** | Planktonzilla | plain CL | 32 |
| `rd_r32` | **LoRA** | Planktonzilla | `ranked-dedup` (RINCE, distance sim) | 32 |
| `rd_r64` | **LoRA** | Planktonzilla | `ranked-dedup` | 64 |

Two clean contrasts among the LoRA runs (backbone/geometry/no-proj all fixed):
- **`e0c` vs `rd_r32`** — *pure loss ablation* (plain CL vs ranked-dedup, both r=32).
- **`rd_r32` vs `rd_r64`** — *rank effect* (ranked-dedup held, r=32 vs r=64).

**Note on data:** the embeddings are of **IFCB** taxonomy strings (145 classes), while all adaptations were trained on **Planktonzilla** — a cross-dataset transfer measurement. No images are used; it embeds 145 *strings*. (`rd_r64` is an `-inprogress`/`_best` checkpoint, not a finished run — treat it as indicative.)

**Failure mode:** sibling species collapsing to near-identical vectors (cosine → 1.0) → the diffusion model can't render them differently.

**Go / no-go per model:** `intra-genus mean` well below `between-genus mean` (large gap), `intra-genus max` under ~1.0.

Kernel: the `DiT` conda env. LoRA embeddings are produced separately in `dino_plankton` (peft + the `hyperbolic-plankton` repo) and saved to `.npz`; this notebook only *loads* them.

In [ ]:
import numpy as np
import pandas as pd
import torch
import open_clip
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path
from itertools import combinations

RECORDS_CSV = "/scratch/datasets/other/IFCB_FishNet_Format/anns/ifcb_records.csv"
IMAGES_DIR  = "/scratch/datasets/other/IFCB_FishNet_Format/Images"

# Live-embeddable CLIP text encoders (loaded in-notebook). Same backbone (OpenCLIP ViT-B/16)
# across all models, so the comparison isolates the adaptation method / loss.
CLIP_MODELS = {
    "plain": ("ViT-B-16", "openai"),    # frozen, no plankton exposure
    "pz":    ("hf-hub:project-oceania/CLIP-ViT-B-16.openai-pt.planktonzilla-pt", None),  # full FT
}

# Precomputed LoRA embeddings (dino_plankton env). Each model has TWO variants:
#   cumulative        -> lineage to species only  (._hierarchical_ / older _text_ npz)
#   cumulative_morpho -> lineage + morphotype suffix  (._morpho_ npz)  <- parity w/ old run
# npz keys: clip_emb_species (the condition), folder. (Coarse null lives in the same file.)
PRECOMPUTED_NPZ = {
    "e0c": {
        "cumulative":        "ifcb_e0c_text_embeddings.npz",
        "cumulative_morpho": "ifcb_e0c_morpho_embeddings.npz",
    },
    "rd_r32": {
        "cumulative":        "ifcb_rd32_hierarchical_embeddings.npz",
        "cumulative_morpho": "ifcb_rd32_morpho_embeddings.npz",
    },
    "rd_r64": {
        "cumulative":        "ifcb_rankeddedup_text_embeddings.npz",
        "cumulative_morpho": "ifcb_rd64_morpho_embeddings.npz",
    },
}

# Which npz key holds the (145, dim) embedding matrix — differs across the older files.
def _emb_key(z):
    for k in ("clip_emb_species", "clip_emb"):
        if k in z:
            return k
    raise KeyError("no embedding array in npz")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("live CLIP models:", list(CLIP_MODELS))
for m, variants in PRECOMPUTED_NPZ.items():
    for v, p in variants.items():
        print(f"precomputed {m:8s} {v:18s}: {p}  (exists: {Path(p).exists()})")

## 1. Build the class list + taxonomy

The 145 classes are the image subfolders (same order `IFCBTrainDataset` assigns class indices: `sorted(data_path.iterdir())`). We join each folder to its taxonomy row in `ifcb_records.csv`. Many coarse taxa have blank Genus/species (e.g. *Acantharia*), which we handle explicitly.

In [2]:
records = pd.read_csv(RECORDS_CSV)

# One taxonomy row per Folder (records has one row per image; take the first).
RANKS = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "species"]
tax = records.groupby("Folder")[RANKS].first()

# Class order exactly as the dataset assigns indices.
folders = sorted([p.name for p in Path(IMAGES_DIR).iterdir() if p.is_dir()])
print(f"{len(folders)} class folders")

missing = [f for f in folders if f not in tax.index]
if missing:
    print(f"WARNING: {len(missing)} folders have no taxonomy row:", missing[:10])

df = tax.reindex(folders).copy()
df.index.name = "Folder"
df = df.reset_index()
df["class_idx"] = range(len(df))
df.head(10)

145 class folders


,Folder,Kingdom,Phylum,Class,Order,Family,Genus,species,class_idx
0,Acantharia,Chromista,Radiozoa,Acantharia,None,None,None,None,0
1,Acanthoica_quattrospina,Chromista,Haptophyta,Coccolithophyceae,Syracosphaerales,Rhabdosphaeraceae,Acanthoica,quattrospina,1
2,Akashiwo_sanguinea,Chromista,Myzozoa,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Akashiwo,sanguinea,2
3,Alexandrium_spp,Chromista,Myzozoa,Dinophyceae,Gonyaulacales,Pyrocystaceae,Alexandrium,None,3
4,Amphidinium_crassum,Chromista,Myzozoa,Dinophyceae,Amphidiniales,Amphidiniaceae,Amphidinium,crassum,4
5,Amphidinium_sphenoides,Chromista,Myzozoa,Dinophyceae,Amphidiniales,Amphidiniaceae,Amphidinium,sphenoides,5
6,Apedinella,Chromista,Ochrophyta,Dictyochophyceae,Pedinellales,Actinomonadaceae,Apedinella,None,6
7,Askenasia,Chromista,Ciliophora,Litostomatea,Cyclotrichiida,Mesodiniidae,Askenasia,None,7
8,Asterompalus_flabellatus,Chromista,Heterokontophyta,Bacillariophyceae,Asterolamprales,Asterolampraceae,Asteromphalus,flabellatus,8
9,Asteromphalus_sarcophagus,Chromista,Heterokontophyta,Bacillariophyceae,Asterolamprales,Asterolampraceae,Asteromphalus,sarcophagus,9


In [3]:
# How complete is each rank? (blanks at coarse taxa are expected)
for r in RANKS:
    n = df[r].notna().sum()
    print(f"{r:9s}: {n:3d}/{len(df)} populated, {df[r].nunique()} unique")

Kingdom  : 141/145 populated, 4 unique
Phylum   : 141/145 populated, 12 unique
Class    : 135/145 populated, 14 unique
Order    : 132/145 populated, 41 unique
Family   : 131/145 populated, 64 unique
Genus    : 131/145 populated, 83 unique
species  :  83/145 populated, 68 unique


## 2. Build the conditioning strings (4 variants)

The string you feed CLIP is a design lever. We test four so you can *see* which best separates species:

| variant | example | intent |
|---|---|---|
| `finest` | `Thalassiosira` | just the most specific populated rank — minimal |
| `genus_species` | `Thalassiosira` (genus) + species epithet if present | the natural binomial |
| `cumulative` | `Bacillariophyta Coscinodiscophyceae ... Thalassiosira` | full path — encodes containment, ties to your hyperbolic/cumulative-text findings |
| `prompted` | `a microscopy image of the plankton Thalassiosira` | pushes strings toward CLIP's caption distribution |

In [ ]:
import difflib

def finest_name(row):
    """Most specific non-empty rank (falls back to the folder name)."""
    for r in reversed(RANKS):
        v = row[r]
        if isinstance(v, str) and v.strip():
            return v.strip()
    return row["Folder"]

def cumulative_name(row):
    parts = [str(row[r]).strip() for r in RANKS if isinstance(row[r], str) and row[r].strip()]
    return " ".join(parts) if parts else row["Folder"]

def genus_species_name(row):
    g = row["Genus"]; s = row["species"]
    parts = [str(x).strip() for x in (g, s) if isinstance(x, str) and str(x).strip()]
    return " ".join(parts) if parts else finest_name(row)

def _fuzzy_eq(a, b, thr=0.8):
    if not a or not b:
        return False
    return difflib.SequenceMatcher(None, a, b).ratio() >= thr

def morphotype_suffix(row):
    """Trailing folder tokens beyond the (fuzzily-matched) genus + species epithet.
    '' for folders that are just genus[/species] or non-taxonomic (no genus in CSV).
    Fuzzy matching absorbs folder-vs-CSV spelling drift (Cerautulina~Cerataulina), so only
    the true distinguishing suffix (single/double/chain/var .../spp) survives.
    This is what the previous LabelEmbedder run distinguished via per-folder class index."""
    folder = row["Folder"]
    toks = folder.lower().split("_")
    g = str(row["Genus"]).strip().lower() if isinstance(row["Genus"], str) else ""
    s = str(row["species"]).strip().lower() if isinstance(row["species"], str) else ""
    s_ep = s.split()[-1] if s else ""
    i = 0
    if i < len(toks) and g and _fuzzy_eq(toks[i], g):
        i += 1
    if i < len(toks) and s_ep and _fuzzy_eq(toks[i], s_ep):
        i += 1
    if not g:              # coarse taxon / junk class -> leave lineage as-is
        return ""
    return " ".join(toks[i:])

df["finest"]             = df.apply(finest_name, axis=1)
df["genus_species"]      = df.apply(genus_species_name, axis=1)
df["cumulative"]         = df.apply(cumulative_name, axis=1)
df["prompted"]           = "a microscopy image of the plankton " + df["finest"]
df["morpho_suffix"]      = df.apply(morphotype_suffix, axis=1)
# cumulative + morphotype: the lineage with the folder's distinguishing token appended, so
# same-species morphotype folders (single/double/chain) get DISTINCT strings — parity with
# the previous FineDiffusion run, which conditioned per-folder.
df["cumulative_morpho"]  = df.apply(
    lambda r: f"{r['cumulative']} {r['morpho_suffix']}".strip() if r["morpho_suffix"] else r["cumulative"],
    axis=1)

VARIANTS = ["finest", "genus_species", "cumulative", "prompted", "cumulative_morpho"]
print(f"{(df['morpho_suffix'] != '').sum()} folders get a morphotype suffix; "
      f"distinct cumulative_morpho strings: {df['cumulative_morpho'].nunique()} "
      f"(vs cumulative: {df['cumulative'].nunique()})")
df[df["morpho_suffix"] != ""][["Folder", "cumulative_morpho"]].head(12)

## 3. Embed all classes with each text encoder

We embed the four string variants with `plain` and `pz` live, and load the LoRA checkpoints (`e0c`, `ranked_dd`) from their `.npz` files. Result: `emb_by_model[model][variant] -> (145, dim)`, L2-normalized so dot product = cosine.

**LoRA note:** the LoRA embeddings were generated on the `cumulative` strings only (the lineage string those models were trained with), so those entries have just the `cumulative` key. The cross-model comparison below therefore uses `cumulative` — the variant we'd actually feed the conditioner.

In [ ]:
def load_clip(spec, pretrained):
    if spec.startswith("hf-hub:"):
        m, _, _ = open_clip.create_model_and_transforms(spec)
        tok = open_clip.get_tokenizer(spec)
    else:
        m, _, _ = open_clip.create_model_and_transforms(spec, pretrained=pretrained)
        tok = open_clip.get_tokenizer(spec)
    return m.eval().to(device), tok

@torch.no_grad()
def embed_with(model, tokenizer, strings):
    toks = tokenizer(list(strings)).to(device)
    feats = model.encode_text(toks).float()
    feats = feats / feats.norm(dim=-1, keepdim=True)   # L2 normalize -> cosine == dot
    return feats.cpu().numpy()

# --- live models: plain + pz, all string variants (incl. cumulative_morpho) ---
emb_by_model = {}
for key, (spec, pretrained) in CLIP_MODELS.items():
    model, tokenizer = load_clip(spec, pretrained)
    emb_by_model[key] = {v: embed_with(model, tokenizer, df[v].tolist()) for v in VARIANTS}
    print(f"{key:8s}: dim {emb_by_model[key]['cumulative'].shape[1]}  ({spec})")
    del model
    torch.cuda.empty_cache()

# --- precomputed LoRA checkpoints: load each variant, align to df row order by Folder ---
def _load_npz_aligned(npz_path):
    z = np.load(npz_path, allow_pickle=True)
    E = z[_emb_key(z)]
    by_folder = {f: E[i] for i, f in enumerate(z["folder"])}
    miss = [f for f in df["Folder"] if f not in by_folder]
    if miss:
        print(f"  WARNING [{npz_path}]: {len(miss)} folders missing:", miss[:5])
    A = np.stack([by_folder[f] for f in df["Folder"]])
    return A / np.linalg.norm(A, axis=1, keepdims=True)

for key, variants in PRECOMPUTED_NPZ.items():
    emb_by_model.setdefault(key, {})
    for v, npz_path in variants.items():
        if not Path(npz_path).exists():
            print(f"{key:8s} {v:18s}: SKIPPED — {npz_path} not found")
            continue
        emb_by_model[key][v] = _load_npz_aligned(npz_path)
    dims = {v: emb_by_model[key][v].shape[1] for v in emb_by_model[key]}
    print(f"{key:8s}: loaded {list(emb_by_model[key])}  dim {list(dims.values())[0]}")

MODELS = list(emb_by_model)
print("\nmodels available for comparison:", MODELS)

## 4. THE diagnostic — intra-genus vs between-genus cosine, per model & variant

We compare each model on **two cumulative-lineage variants** (both contain the full `Kingdom…species` path — they differ *only* by the appended leaf token):
- `cumulative` — lineage to species.
- `cumulative_morpho` — the same lineage **+ the folder's morphotype suffix** (`… gravida single`), giving same-species morphotype folders distinct strings. This is parity with the previous FineDiffusion run's per-folder conditioning.

Metrics per (model, variant):
- **intra-genus mean/max** — pairwise cosine among genus-mates. `max = 1.0` = a pair the encoder cannot separate at all.
- **between-genus mean** — baseline across genera.
- **gap = intra − between**.
- **pairs>0.999** — count of genus-mate pairs at near-identical cosine (the hard collisions morpho is meant to break).

In [ ]:
# Genus groups (shared across all models — geometry-independent).
has_genus = df["Genus"].apply(lambda v: isinstance(v, str) and v.strip() != "")
genus_of = df["Genus"].where(has_genus)
genus_groups = {g: idxs.tolist() for g, idxs in df.index[has_genus].to_series().groupby(genus_of[has_genus])}
multi_genera = {g: idx for g, idx in genus_groups.items() if len(idx) >= 2}
gidx = df.index[has_genus].tolist()
print(f"{has_genus.sum()} classes have a genus; "
      f"{len(multi_genera)} genera with >=2 classes "
      f"({sum(len(v) for v in multi_genera.values())} classes in the intra-genus test)")

def cos(a, b):
    return float(np.dot(a, b))

def spread_stats(E):
    """intra/between-genus cosine stats for one (145, dim) embedding matrix."""
    intra = [cos(E[i], E[j]) for idx in multi_genera.values() for i, j in combinations(idx, 2)]
    rng = np.random.default_rng(0)
    between = []
    for _ in range(5000):
        i, j = rng.choice(gidx, 2, replace=False)
        if genus_of[i] != genus_of[j]:
            between.append(cos(E[i], E[j]))
    intra, between = np.array(intra), np.array(between)
    n_collapsed = int((intra > 0.999).sum())
    return {
        "intra_genus_mean": intra.mean(),
        "intra_genus_max": intra.max(),
        "between_genus_mean": between.mean(),
        "gap(intra-between)": intra.mean() - between.mean(),
        "pairs>0.999": n_collapsed,
    }

# Compare each model on BOTH variants: cumulative (species-level) vs cumulative_morpho.
# The morpho column is the parity-with-old-run condition (per-folder distinguishing suffix).
COMPARE_VARIANTS = ["cumulative", "cumulative_morpho"]
rows = []
for m in MODELS:
    for v in COMPARE_VARIANTS:
        if v not in emb_by_model[m]:
            continue
        rows.append({"model": m, "variant": v, **spread_stats(emb_by_model[m][v])})
res = pd.DataFrame(rows).set_index(["model", "variant"]).round(3)
res

**Reading the per-model table:** the model with the **largest `gap`** and **lowest `between_genus_mean`** separates the taxonomy best. But watch `intra_genus_max`: if it's 1.000 for *every* model, there exist class pairs that **no** text encoder separates — inspect them below (they're almost always same-species *morphotype* folders, e.g. `_single`/`_double`/`_chain`, whose taxonomy strings are identical, so no text model can tell them apart).

In [ ]:
# Top collapsed pairs per model, for BOTH variants — does appending the morphotype break
# the same-species collisions?
for m in MODELS:
    for v in COMPARE_VARIANTS:
        if v not in emb_by_model[m]:
            continue
        E = emb_by_model[m][v]
        pairs = [(cos(E[i], E[j]), df.loc[i, "Folder"], df.loc[j, "Folder"])
                 for idx in multi_genera.values() for i, j in combinations(idx, 2)]
        pairs.sort(reverse=True)
        n = sum(1 for c, _, _ in pairs if c > 0.999)
        print(f"\n=== {m} / {v}   [{n} pairs at cos>0.999] ===")
        for c, a, b in pairs[:4]:
            print(f"  {c:.3f}  {a}  ~  {b}")

## 4b. Goal-relevant metrics: nearest-neighbour separation & hierarchy faithfulness

Mean/max intra-genus cosine (§4) is a *mean-pairwise* statistic — and our prior work found that classification is driven by **nearest-neighbour** separation + margin, not mean separability (mean can mislead). So we add two metrics that better track the conditioning goal:

- **nn-sep** — for each class, cosine to its **nearest other class** (the closest confuser). **Lower = better separated.** `nn_sep_max = 1.0` means at least one class still has an essentially-identical neighbour. This is the discriminability metric.
- **hier-genus AUC** — P(a same-genus pair is closer than a different-genus pair). **1.0 = perfectly hierarchy-faithful.** This is the hierarchy-transfer metric (what TaxaDiffusion's progressive conditioning relies on).

These two often *disagree*: an encoder that spreads everything apart (good nn-sep) need not respect hierarchy (lower AUC), and vice-versa — the discrimination↔hierarchy tension. Which matters more is a generative question, not settled by embeddings alone.

In [ ]:
genus_of = [df.loc[i, "Genus"] if isinstance(df.loc[i, "Genus"], str) and df.loc[i, "Genus"].strip()
            else None for i in range(len(df))]

def nn_separation(E):
    """Per-class cosine to the nearest OTHER class. Returns (mean, max). Lower = better."""
    S = E @ E.T
    np.fill_diagonal(S, -9.0)
    nn = S.max(axis=1)
    return float(nn.mean()), float(nn.max())

def hier_genus_auc(E):
    """P(same-genus pair cosine > different-genus pair cosine). 1.0 = perfect hierarchy."""
    correct = tot = 0
    for gi, idx in multi_genera.items():
        others = [j for j in range(len(E)) if genus_of[j] != gi]
        for a in idx:
            same = np.array([cos(E[a], E[b]) for b in idx if b != a])
            diff = np.array([cos(E[a], E[b]) for b in others])
            if not same.size or not diff.size:
                continue
            correct += int((same[:, None] > diff[None, :]).sum())
            tot += same.size * diff.size
    return correct / tot if tot else float("nan")

rows = []
for m in MODELS:
    for v in COMPARE_VARIANTS:
        if v not in emb_by_model[m]:
            continue
        E = emb_by_model[m][v]
        nn_mean, nn_max = nn_separation(E)
        rows.append({
            "model": m, "variant": v,
            "nn_sep_mean": round(nn_mean, 3),   # lower = better separated
            "nn_sep_max": round(nn_max, 3),
            "hier_genus_auc": round(hier_genus_auc(E), 3),  # higher = more hierarchy-faithful
        })
pd.DataFrame(rows).set_index(["model", "variant"])

## 5. Does the coarse hierarchy survive? (intra-phylum)

FineDiffusion's superclass guidance needs the *phylum* level to be coherent. Same test, grouping by Phylum instead of Genus.

In [ ]:
has_phy = df["Phylum"].apply(lambda v: isinstance(v, str) and v.strip() != "")
phy_of = df["Phylum"].where(has_phy)
phy_groups = {p: idxs.tolist() for p, idxs in df.index[has_phy].to_series().groupby(phy_of[has_phy])}
phy_groups = {p: idx for p, idx in phy_groups.items() if len(idx) >= 2}

rows = []
for m in MODELS:
    if "cumulative" not in emb_by_model[m]:
        continue
    E = emb_by_model[m]["cumulative"]
    intra = [cos(E[i], E[j]) for idx in phy_groups.values() for i, j in combinations(idx, 2)]
    rows.append({"model": m, "intra_phylum_mean": np.mean(intra), "n_pairs": len(intra)})
pd.DataFrame(rows).set_index("model").round(3)

## 6. Visual: PCA per model, colored by Phylum (side by side)

Same 145 classes (`cumulative` strings) through each encoder. Better model = phyla form tighter, more separated blobs. Watch how much the cloud opens up from `plain` → `pz` → `e0c`.

## 7. Verdict

All encoders share the ViT-B/16 backbone, so differences are the adaptation method / loss. Reading §4 (cosine), §4b (nn-sep + hierarchy), and the collapsed pairs together:

**1. Morphotype append does its one job.** `cumulative` → `cumulative_morpho` (same lineage + leaf token) drops the hard same-species collisions (`pairs>0.999`) from ~22 to ~0 for every LoRA encoder — parity with the previous per-folder run. It barely moves nn-sep / hierarchy (a leaf-level tweak), and `pz` (full-FT) resists it (keeps ~7), having learned to ignore sub-species text.

**2. The two goal-relevant metrics rank the encoders *oppositely* — this is the real finding.**
- **nn-sep (discriminability, lower=better):** `e0c` wins (~0.72) — each class's nearest confuser is far. `rd_r32/64` worst (~0.90–0.92) — ranked-dedup's tight genus clusters crowd siblings.
- **hier-genus AUC (hierarchy faithfulness, higher=better):** `rd_r64` perfect (1.000), `rd_r32` ~0.999 — ranked-dedup builds near-flawless hierarchy. `e0c` worst (~0.97).

`e0c` and `rd` are **not** better/worse versions of each other — they sit at opposite ends of a **discrimination ↔ hierarchy** trade-off (the containment-vs-discrimination tension, measured directly).

**3. Which to condition on cannot be decided from embeddings.** If per-class discriminability dominates the generative goal → `e0c`. If hierarchy-transfer to rare classes dominates (TaxaDiffusion's thesis) → `rd`. Only a **generation experiment** (per-rank FID / LPIPS / BioCLIP on generated images) settles it. The `cumulative` cosine `gap` metric alone is *not* a valid selector — it scored e0c and rd near-ties while they encode opposite geometries.

**Practical:** the hybrid `ClipEmbedder` (CLIP text + per-class code) is robust to this either way — the per-class code supplies discriminability regardless of which encoder's geometry you pick, and matches the old run's per-folder behaviour. Embeddings for both variants of every model are saved below.

In [ ]:
# All-pairs cosine distribution per model — how wide a cone does each encoder use?
models_plot = [m for m in MODELS if "cumulative" in emb_by_model[m]]
fig, axes = plt.subplots(1, len(models_plot), figsize=(5 * len(models_plot), 3.2), sharey=True, squeeze=False)
for ax, m in zip(axes[0], models_plot):
    E = emb_by_model[m]["cumulative"]
    allc = [cos(E[i], E[j]) for i, j in combinations(range(len(E)), 2)]
    ax.hist(allc, bins=40, color="steelblue")
    ax.set_title(f"{m}  (mean {np.mean(allc):.2f})"); ax.set_xlabel("cosine")
    ax.axvline(np.mean(allc), color="r", ls="--", lw=1)
    ax.set_xlim(-0.2, 1.05)
axes[0][0].set_ylabel("pair count")
plt.suptitle("All-pairs cosine distribution per model (cumulative) — lower/wider = more spread")
plt.tight_layout(); plt.show()

## 7. Verdict

All three encoders are the **same ViT-B/16 backbone**, so differences are purely the adaptation method (none / full-FT / LoRA). Read the per-model table (§4) with the collapsed-pairs list (§4b) and the PCA/histograms:

- **Best model = largest `gap` + lowest `between_genus_mean`.** Expect `plain` (frozen) < `pz` (full-FT) < `e0c` (LoRA): Planktonzilla adaptation spreads the taxonomy far wider than frozen CLIP, and E0c — contrastively trained on the lineage strings — spreads it most, so genuinely different species/genera become well separated.
- **The residual wall:** `intra_genus_max` stays **1.000 for every model**, driven by same-species *morphotype* folders (`_single`/`_double`/`_chain`, etc.). Their taxonomy strings are identical, so **no text encoder — not even E0c — can separate them.** That distinction is *visual*, not lexical.

**Implication for the FineDiffusion conditioner:** use **E0c `cumulative` text embeddings** as the conditioning prior (best taxonomic separation + zero-shot generalization), and add a tie-breaker only for the handful of morphotype folders text can't reach — either a small learned per-class code or an image-CLIP embedding. That's the hybrid `ClipEmbedder`, built on E0c rather than frozen CLIP.

Below we save each model's `cumulative` embeddings for downstream use:

In [ ]:
# Save each model's embeddings for downstream use, both variants where available.
# The cumulative_morpho set is the one that matches the previous FineDiffusion run's
# per-folder conditioning (morphotype-distinguished).
for m in MODELS:
    for v in COMPARE_VARIANTS:
        if v not in emb_by_model[m]:
            continue
        E = emb_by_model[m][v].astype(np.float32)
        path = f"ifcb_text_embeddings_{m}_{v}.npz"
        np.savez(path,
                 class_idx=df["class_idx"].values,
                 folder=df["Folder"].values,
                 string=df[v].values,
                 clip_emb=E,      # L2-normalized, aligned to class_idx
                 model=m, variant=v)
        print(f"saved {path}  -> {E.shape}")